# Image Region Selection for xradio Images

This notebook demonstrates the extended CRTF selection features available when working with
xradio-format sky images — five-dimensional DataArrays with dims
`(time, frequency, polarization, l, m)` and the corresponding named coordinates.

For basic selection on generic 2-D arrays using pixel coordinates, see
**`generic_image_selection.ipynb`** in the same directory.

## Features covered

| Section | Feature | Coordinate required |
|---|---|---|
| [lm-mode shapes](#lm-mode-shapes) | Angular offsets in arcsec / arcmin | `l`, `m` |
| [world-mode shapes](#world-mode-shapes) | Absolute RA / Dec | `right_ascension`, `declination` |
| [`range=`](#range-selection) | Frequency, velocity, or channel | `frequency` / `velocity` |
| [`corr=`](#corr-selection) | Polarization (Stokes) | `polarization` |
| [`time=`](#time-selection) | Time range (MJD / ISO) | `time` |

In [ ]:
import math
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from astroviper.distributed.image_analysis.selection import select_mask
from astroviper.utils.plotting import generate_plot

## Building a representative sky DataArray

xradio sky images have five dimensions: `time`, `frequency`, `polarization`, `l`, `m`.
The spatial dimensions `l` and `m` are angular offsets (in **radians**) east and north
of the image reference direction.  In practice the values are tiny numbers — a
typical 100-arcsec-wide image spans roughly ±2.4 × 10⁻⁴ rad in each axis.

The helper below builds a small but realistic-looking sky DataArray that we will use
throughout this notebook.

In [ ]:
def make_sky(
    n_l: int = 100,
    n_m: int = 100,
    cell_arcsec: float = 1.0,
    with_radec: bool = False,
    ra0_deg: float = 15.0,
    dec0_deg: float = 30.0,
    n_freq: int = 1,
    freq_start_ghz: float = 1.4,
    freq_step_ghz: float = 0.1,
    vel_start: float = 0.0,
    vel_step: float = 1e4,
    pols: list = None,
    n_time: int = 1,
    time_start_mjd: float = 60000.0,
    time_step_mjd: float = 1.0,
) -> xr.DataArray:
    """Build a synthetic xradio-like sky DataArray.

    The image contains a bright Gaussian blob near the centre so that
    the selected regions are easy to see.

    Parameters
    ----------
    n_l, n_m : int
        Number of pixels along l and m.
    cell_arcsec : float
        Pixel size in arcseconds.
    with_radec : bool
        When True, add ``right_ascension`` and ``declination`` 2-D coords
        using a tangent-plane projection centred on (ra0_deg, dec0_deg).
        Required for world-mode shape examples.
    ra0_deg, dec0_deg : float
        Reference direction in decimal degrees (used only when with_radec=True).
    n_freq : int
        Number of frequency channels.  1 gives a single channel at freq_start_ghz.
    freq_start_ghz : float
        First channel centre frequency in GHz.
    freq_step_ghz : float
        Channel spacing in GHz.
    vel_start : float
        Radial velocity of the first channel in m/s.
    vel_step : float
        Velocity step between channels in m/s.
    pols : list[str] | None
        Polarization labels.  Defaults to ``['I']``.
    n_time : int
        Number of time steps.
    time_start_mjd : float
        Start time in Modified Julian Days.
    time_step_mjd : float
        Time step in days.

    Returns
    -------
    xr.DataArray
        Shape (n_time, n_freq, n_pol, n_l, n_m) with dims
        (time, frequency, polarization, l, m).
    """
    if pols is None:
        pols = ["I"]

    arcsec = math.pi / (180 * 3600)          # 1 arcsec in radians
    cell = cell_arcsec * arcsec

    # l/m coords: centred on zero, spacing = cell_arcsec
    l_rad = (np.arange(n_l) - (n_l - 1) / 2.0) * cell
    m_rad = (np.arange(n_m) - (n_m - 1) / 2.0) * cell

    # A simple Gaussian blob: peak = 1.0, sigma = 10 arcsec
    ll, mm = np.meshgrid(l_rad, m_rad, indexing="ij")
    sigma = 10 * arcsec
    blob = np.exp(-(ll**2 + mm**2) / (2 * sigma**2))

    n_pol = len(pols)
    data = np.broadcast_to(
        blob[np.newaxis, np.newaxis, np.newaxis, :, :],
        (n_time, n_freq, n_pol, n_l, n_m),
    ).copy()

    freq_hz  = (freq_start_ghz + np.arange(n_freq) * freq_step_ghz) * 1e9
    vel_ms   = vel_start + np.arange(n_freq) * vel_step
    time_mjd = time_start_mjd + np.arange(n_time) * time_step_mjd

    coords = {
        "time":         (["time"],         time_mjd),
        "frequency":    (["frequency"],    freq_hz),
        "velocity":     (["frequency"],    vel_ms),
        "polarization": (["polarization"], pols),
        "l":            (["l"],            l_rad),
        "m":            (["m"],            m_rad),
    }

    if with_radec:
        ra0  = ra0_deg  * math.pi / 180.0
        dec0 = dec0_deg * math.pi / 180.0
        # Tangent-plane approx: RA offset ≈ l / cos(dec0), Dec offset ≈ m (radians)
        ra_grid  = ra0  + ll / math.cos(dec0)
        dec_grid = dec0 + mm
        coords["right_ascension"] = (["l", "m"], ra_grid)
        coords["declination"]     = (["l", "m"], dec_grid)

    return xr.DataArray(
        data,
        dims=["time", "frequency", "polarization", "l", "m"],
        coords=coords,
    )

sky = make_sky()
sky

The `l` and `m` coordinates carry the physical angular scale.  We will always express
region boundaries in **arcsec** or **arcmin**, and `select_mask` converts them to radians
internally before comparing against the coordinate grids.

The visualisation helper below calls `generate_plot` from `astroviper.utils.plotting`
for each figure — this ensures correct axis-label conventions, the right (x, y) storage
order, and proper colorbar handling.  Two figures are produced for each example:

1. **Sky image** — the intensity map with angular offset axes in arcseconds.
2. **Selected region** — the same image with the mask overlaid: excluded pixels are
   dimmed and the region boundary is drawn in white.

Following the standard astronomical convention for direction-cosine coordinates,
the **l-axis (east) increases to the LEFT** on both figures.

In [ ]:
def plot_sky_with_mask(sky: xr.DataArray, mask: xr.DataArray, title: str = "") -> None:
    """Show a sky image and a mask overlay using generate_plot from astroviper.utils.plotting.

    Two figures are produced:
      1. The plain sky image with properly labelled angular axes.
      2. The same image with the selected region indicated by a white boundary
         and excluded pixels dimmed.

    Parameters
    ----------
    sky : xr.DataArray
        5-D sky DataArray with dims (time, frequency, polarization, l, m).
        The l and m coordinates must be in radians.
    mask : xr.DataArray
        Boolean mask from select_mask(), aligned with *sky*.
    title : str
        Short description of the region shown in the second plot title.

    Notes
    -----
    * l and m are converted from radians to arcseconds for display.
    * The l-axis (east) is inverted so that east is to the LEFT — the standard
      astronomical orientation for images in direction-cosine coordinates.
    * generate_plot handles axis labelling, colorbar, and the correct (x, y)
      storage-order convention automatically via show_world_axes=True.
    """
    arcsec_rad = math.pi / (180 * 3600)

    # Squeeze the 5-D arrays to 2-D (l, m) for plotting
    sky_2d  = sky.isel(time=0, frequency=0, polarization=0)
    mask_2d = mask.squeeze()   # drops all size-1 dims -> (l, m)

    # Angular axes in arcseconds
    l_arcsec = sky_2d.coords["l"].values / arcsec_rad
    m_arcsec = sky_2d.coords["m"].values / arcsec_rad

    _kw = dict(
        show_world_axes=True,
        x_coords=l_arcsec,
        y_coords=m_arcsec,
        cmap="viridis",
        figsize=(5.5, 5.0),
    )

    # --- Panel 1: plain sky image ---
    fig, ax = generate_plot(sky_2d, title="Sky image", **_kw)
    ax.invert_xaxis()   # east (increasing l) is to the LEFT
    ax.set_xlabel("l offset  [arcsec]")
    ax.set_ylabel("m offset  [arcsec]")
    plt.tight_layout()
    plt.show()

    # --- Panel 2: sky image with mask overlay ---
    panel_title = f"Selected region — {title}" if title else "Selected region"
    fig, ax = generate_plot(sky_2d, title=panel_title, **_kw)
    ax.invert_xaxis()
    ax.set_xlabel("l offset  [arcsec]")
    ax.set_ylabel("m offset  [arcsec]")

    mask_np = np.asarray(mask_2d.values, dtype=float)
    # Dim excluded pixels with a semi-transparent grey overlay
    excluded = np.where(mask_np > 0.5, np.nan, 0.6)
    ax.pcolormesh(l_arcsec, m_arcsec, excluded.T, cmap="Greys_r", alpha=0.55, vmin=0, vmax=1)
    # White boundary contour marks the region edge
    ax.contour(l_arcsec, m_arcsec, mask_np.T, levels=[0.5], colors="white", linewidths=1.5)
    plt.tight_layout()
    plt.show()

<a id="lm-mode-shapes"></a>
## lm-mode shapes — angular offsets in arcsec / arcmin

When all shape coordinates are expressed in **arcsec** or **arcmin**, `select_mask`
automatically detects *lm mode* and compares the CRTF distances directly against the
physical `l` and `m` coordinates on the DataArray.  No `coordsys=` keyword is needed.

This is the most natural way to specify regions on a radio-astronomy image: you say
"give me a circle of radius 15 arcsec centred on the image reference direction" and the
code works out which pixels fall inside it.

All center / vertex coordinates are **angular offsets from the reference direction**
(i.e., from l = m = 0).  Positive l is east; positive m is north.

Supported shapes: `circle`, `annulus`, `box`, `centerbox`, `rotbox`, `ellipse`, `poly`.
Each is shown with a short example below.

### circle

Syntax: `circle[[center_l, center_m], radius]`

Selects all pixels within `radius` of the given center.  Here we pick a 15-arcsec
circle centred exactly on the reference direction (l = m = 0), which neatly captures
the Gaussian blob.

In [ ]:
crtf_circle = """
#CRTF
circle[[0arcsec, 0arcsec], 15arcsec]
""".strip()

mask_circle = select_mask(sky, crtf_circle)
print("Selected pixels:", int(mask_circle.values.sum()))
plot_sky_with_mask(sky, mask_circle, title="circle r=15arcsec")

### annulus

Syntax: `annulus[[center_l, center_m], [inner_radius, outer_radius]]`

Selects the ring of pixels whose distance from the center falls between the two radii.
Useful for selecting a background annulus around a source.

In [ ]:
crtf_annulus = """
#CRTF
annulus[[0arcsec, 0arcsec], [20arcsec, 35arcsec]]
""".strip()

mask_annulus = select_mask(sky, crtf_annulus)
print("Selected pixels:", int(mask_annulus.values.sum()))
plot_sky_with_mask(sky, mask_annulus, title="annulus 20–35arcsec")

### centerbox

Syntax: `centerbox[[center_l, center_m], [width, height]]`

Selects a rectangle of `width` × `height` centred at the given position.
The width is along l (east) and the height is along m (north).

Here we select a 30 × 20 arcsec box offset 5 arcsec east and 5 arcsec north of the
reference direction.

In [ ]:
crtf_centerbox = """
#CRTF
centerbox[[5arcsec, 5arcsec], [30arcsec, 20arcsec]]
""".strip()

mask_centerbox = select_mask(sky, crtf_centerbox)
print("Selected pixels:", int(mask_centerbox.values.sum()))
plot_sky_with_mask(sky, mask_centerbox, title="centerbox 30×20arcsec, offset 5arcsec NE")

### box

Syntax: `box[[blc_l, blc_m], [trc_l, trc_m]]`

Selects a rectangle given its bottom-left corner (blc) and top-right corner (trc).
Both corners are angular offsets from the reference direction.

This selects the lower-left quadrant of the image (l < 0, m < 0).

In [ ]:
crtf_box = """
#CRTF
box[[-50arcsec, -50arcsec], [0arcsec, 0arcsec]]
""".strip()

mask_box = select_mask(sky, crtf_box)
print("Selected pixels:", int(mask_box.values.sum()))
plot_sky_with_mask(sky, mask_box, title="box: lower-left quadrant")

### rotbox — rotated rectangle

Syntax: `rotbox[[center_l, center_m], [width, height], pa=<angle>]`

Like `centerbox` but rotated by a position angle.  The angle is specified with one of
two keywords to avoid ambiguity:

- **`pa=<angle>`** — position angle measured from +m (north) toward +l (east).  This is
  the conventional astronomical position angle.
- **`theta_m=<angle>`** — math angle measured from +l (east) toward +m (north).

Angle units: `deg` or `rad`.

Here we select a 40 × 15 arcsec box tilted 30° east of north (pa=30deg).

In [ ]:
crtf_rotbox = """
#CRTF
rotbox[[0arcsec, 0arcsec], [40arcsec, 15arcsec], pa=30deg]
""".strip()

mask_rotbox = select_mask(sky, crtf_rotbox)
print("Selected pixels:", int(mask_rotbox.values.sum()))
plot_sky_with_mask(sky, mask_rotbox, title="rotbox 40×15arcsec, pa=30deg")

### ellipse

Syntax: `ellipse[[center_l, center_m], [semi_major, semi_minor], pa=<angle>]`

The two sizes are the **semi-axes** (half-widths), not full widths.  The position angle
convention (`pa=` or `theta_m=`) is the same as for `rotbox`.

Here the semi-major axis is 20 arcsec along the east–west direction (pa=90deg places
the major axis pointing east).

In [ ]:
crtf_ellipse = """
#CRTF
ellipse[[0arcsec, 0arcsec], [20arcsec, 10arcsec], pa=90deg]
""".strip()

mask_ellipse = select_mask(sky, crtf_ellipse)
print("Selected pixels:", int(mask_ellipse.values.sum()))
plot_sky_with_mask(sky, mask_ellipse, title="ellipse 20×10arcsec semi-axes, pa=90deg")

### poly — arbitrary polygon

Syntax: `poly[[l0, m0], [l1, m1], ..., [lN, mN]]`

Selects pixels inside a polygon defined by a list of vertices (angular offsets).
The polygon is automatically closed (the last vertex connects back to the first).

Here we draw a rough L-shape in the upper portion of the image.

In [ ]:
crtf_poly = (
    "#CRTF\n"
    "poly[[-30arcsec, 10arcsec], [0arcsec, 10arcsec], [0arcsec, 30arcsec],"
    " [20arcsec, 30arcsec], [20arcsec, 40arcsec], [-30arcsec, 40arcsec]]"
)

mask_poly = select_mask(sky, crtf_poly)
print("Selected pixels:", int(mask_poly.values.sum()))
plot_sky_with_mask(sky, mask_poly, title="poly — L-shape")

### arcmin units

All of the examples above use arcsec.  Arcmin works exactly the same way — mix and
match units within a single shape is not allowed, but different shapes in the same CRTF
string may use different units.

Here is a 0.5-arcmin (= 30 arcsec) radius circle to confirm the unit conversion:

In [ ]:
crtf_arcmin = "#CRTF\ncircle[[0arcmin, 0arcmin], 0.5arcmin]"
mask_arcmin = select_mask(sky, crtf_arcmin)

crtf_arcsec = "#CRTF\ncircle[[0arcsec, 0arcsec], 30arcsec]"
mask_arcsec = select_mask(sky, crtf_arcsec)

# The two masks must be identical
np.testing.assert_array_equal(mask_arcmin.values, mask_arcsec.values)
print("0.5arcmin and 30arcsec produce identical masks \u2713")
plot_sky_with_mask(sky, mask_arcmin, title="0.5arcmin = 30arcsec circle")

### Combining lm-mode shapes

Multi-line CRTF works the same as in the pixel-mode notebook.  A leading `+` (or no
prefix) adds the region; a leading `-` subtracts it from the accumulated mask.

Example: select the Gaussian blob core, then punch out an offset circle.

In [ ]:
crtf_combined = """
#CRTF
+circle[[0arcsec, 0arcsec], 25arcsec]
-circle[[5arcsec, -5arcsec], 8arcsec]
""".strip()

mask_combined = select_mask(sky, crtf_combined)
print("Selected pixels:", int(mask_combined.values.sum()))
plot_sky_with_mask(sky, mask_combined, title="25arcsec circle minus 8arcsec core")

<a id="world-mode-shapes"></a>
## World-mode shapes — absolute RA / Dec

When shape coordinates are given as **sexagesimal RA/Dec** strings (`1h0m0.000s`, `+30d0m0.000s`)
or as **decimal degrees** with `coordsys=world`, `select_mask` operates in *world mode*.
Instead of comparing against l/m angular offsets it uses the `right_ascension` and `declination`
2-D coordinates attached to the DataArray and performs the comparison in sky-coordinate space
via astropy's `SkyCoord`.

**Requirements:**
- The DataArray must carry `right_ascension` and `declination` as 2-D coords (one value per pixel, in radians).
- Use `make_sky(with_radec=True)` to build such a DataArray.

**Auto-detection:** sexagesimal RA/Dec tokens (`Hh Mm Ss` / `Dd Mm Ss`) are recognised as
world-mode coordinates automatically — no `coordsys=` keyword needed.  Decimal-degree tokens
(`N.Ndeg`) require an explicit `coordsys=world`.

**Note on Dask:** v1 materialises the RA/Dec grids before calling astropy (which cannot remain
lazy).  This is expected and documented behaviour for world-mode shapes.

Supported shapes: `circle`, `annulus`, `centerbox`, `box`, `rotbox`, `ellipse`, `poly`.

In [ ]:
# Reference direction: RA = 15 deg = 1h0m0.000s, Dec = +30 deg = +30d0m0.000s
RA0_DEG, DEC0_DEG = 15.0, 30.0

sky_w = make_sky(with_radec=True, ra0_deg=RA0_DEG, dec0_deg=DEC0_DEG)
print("right_ascension coord present:", "right_ascension" in sky_w.coords)
print("declination coord present:    ", "declination"     in sky_w.coords)
sky_w

### circle

Syntax: `circle[[RA, Dec], radius]`

The center can be sexagesimal RA/Dec — this is **auto-detected** as world mode.
Here: center = 1h0m0.000s, +30d0m0.000s; radius = 15 arcsec.

In [ ]:
crtf_w_circle = "#CRTF\ncircle[[1h0m0.000s,+30d0m0.000s], 15arcsec]"
mask_w_circle = select_mask(sky_w, crtf_w_circle)
print("Selected pixels (spatial):", int(mask_w_circle.squeeze().values.sum()))
plot_sky_with_mask(sky_w, mask_w_circle, title="world-mode circle r=15arcsec")

### centerbox

Syntax: `centerbox[[RA, Dec], [width, height]]`

Width and height are angular sizes in arcsec (or arcmin).
The sexagesimal center is auto-detected as world mode.

In [ ]:
crtf_w_cbox = "#CRTF\ncenterbox[[1h0m0.000s,+30d0m0.000s], [30arcsec, 20arcsec]]"
mask_w_cbox = select_mask(sky_w, crtf_w_cbox)
print("Selected pixels (spatial):", int(mask_w_cbox.squeeze().values.sum()))
plot_sky_with_mask(sky_w, mask_w_cbox, title="world-mode centerbox 30×20arcsec")

### box — decimal degrees with coordsys=world

Syntax: `box[[RA_blc, Dec_blc], [RA_trc, Dec_trc]] coordsys=world`

Decimal-degree tokens (`N.Ndeg`) are ambiguous without `coordsys=` (they could be angular offsets
or world coordinates).  The `coordsys=world` keyword resolves the ambiguity.

Here BLC = (14.996 deg, 29.996 deg) and TRC = (15.004 deg, 30.004 deg) define a ~28.8 × 28.8 arcsec
box centred on the reference direction.

In [ ]:
crtf_w_box = "#CRTF\nbox[[14.996deg, 29.996deg], [15.004deg, 30.004deg]] coordsys=world"
mask_w_box = select_mask(sky_w, crtf_w_box)
print("Selected pixels (spatial):", int(mask_w_box.squeeze().values.sum()))
plot_sky_with_mask(sky_w, mask_w_box, title="world-mode box ~28.8×28.8arcsec")

### rotbox — rotated box

Syntax: `rotbox[[RA, Dec], [width, height], pa=<angle>]`

Position angle convention is the same as lm-mode: `pa=` is east of north (conventional
astronomical PA).  Both sexagesimal center (auto-detected) and arcsec sizes are used.

In [ ]:
crtf_w_rotbox = "#CRTF\nrotbox[[1h0m0.000s,+30d0m0.000s], [40arcsec, 15arcsec], pa=45deg]"
mask_w_rotbox = select_mask(sky_w, crtf_w_rotbox)
print("Selected pixels (spatial):", int(mask_w_rotbox.squeeze().values.sum()))
plot_sky_with_mask(sky_w, mask_w_rotbox, title="world-mode rotbox 40×15arcsec pa=45deg")

### ellipse

Syntax: `ellipse[[RA, Dec], [semi_major, semi_minor], pa=<angle>]`

Both sizes are semi-axes (half-widths).  PA convention is the same as rotbox.

In [ ]:
crtf_w_ellipse = "#CRTF\nellipse[[1h0m0.000s,+30d0m0.000s], [20arcsec, 10arcsec], pa=0deg]"
mask_w_ellipse = select_mask(sky_w, crtf_w_ellipse)
print("Selected pixels (spatial):", int(mask_w_ellipse.squeeze().values.sum()))
plot_sky_with_mask(sky_w, mask_w_ellipse, title="world-mode ellipse 20×10arcsec semi-axes")

### poly — arbitrary polygon with decimal degrees

Polygon vertices can be decimal degrees with `coordsys=world`.
Here we draw a triangle centred on the reference direction.

Vertices are provided on a **single Python line** (the CRTF parser splits on newlines, so
multi-line vertex lists are not supported).

In [ ]:
# Triangle: base ±18 arcsec east/west of centre, apex 36 arcsec north
# 18 arcsec = 0.005 deg in Dec; RA correction for cos(Dec): 0.005/cos(30°) ≈ 0.00577 deg
crtf_w_poly = (
    "#CRTF\n"
    "poly[[14.994deg, 29.995deg], [15.006deg, 29.995deg], [15.000deg, 30.010deg]]"
    " coordsys=world"
)
mask_w_poly = select_mask(sky_w, crtf_w_poly)
print("Selected pixels (spatial):", int(mask_w_poly.squeeze().values.sum()))
plot_sky_with_mask(sky_w, mask_w_poly, title="world-mode poly (triangle)")

<a id="range-selection"></a>
## `range=` — frequency, velocity, and channel selection

The `range=` keyword restricts the mask to a contiguous range along the frequency axis.
Three token families are supported:

| Family | Example token | Matched coordinate |
|---|---|---|
| frequency | `1.2GHz` | `frequency` (Hz) |
| velocity | `20km/s` or `20000m/s` | `velocity` (m/s) |
| channel | `3chan` | integer channel index |

Rules:
- Both endpoints in a `range=` must belong to the **same family** (mixing raises `ValueError`).
- The `frequency` coordinate must be present for frequency/channel ranges.
- The `velocity` coordinate must be present for velocity ranges.
- A range that does not overlap the available channels returns an **all-False** mask.

`range=` combines with a shape on the same line: the effective mask is the AND of the spatial
shape mask and the frequency/channel mask.

In [ ]:
# 10-channel sky: frequencies 1.0–1.9 GHz, velocities 0–90 km/s
sky_freq = make_sky(
    n_freq=10,
    freq_start_ghz=1.0,
    freq_step_ghz=0.1,
    vel_start=0.0,
    vel_step=1e4,          # 10 km/s steps
)
print("Frequencies (GHz):", sky_freq.coords["frequency"].values / 1e9)
print("Velocities (km/s):", sky_freq.coords["velocity"].values / 1e3)

### Channel range

`range=[Nchan, Mchan]` selects channels by integer index (0-based, inclusive).

In [ ]:
crtf_chan = "#CRTF\ncircle[[0arcsec, 0arcsec], 20arcsec], range=[2chan, 5chan]"
mask_chan = select_mask(sky_freq, crtf_chan)

freq_selected = mask_chan.any(dim=["time", "polarization", "l", "m"])
print("Channels selected (indices):", list(np.where(freq_selected.values)[0]))
print("Frequencies (GHz):          ", list(sky_freq.coords["frequency"].values[freq_selected.values] / 1e9))

# Plot spatial mask for the first selected channel (index 2)
plot_sky_with_mask(sky_freq, mask_chan.isel(frequency=2), title="channel range 2–5, freq=1.2GHz slice")

### Frequency range

`range=[f1, f2]` with units `GHz`, `MHz`, or `Hz` selects channels whose centre frequency
falls within the range (inclusive).

In [ ]:
crtf_freq = "#CRTF\ncircle[[0arcsec, 0arcsec], 20arcsec], range=[1.2GHz, 1.5GHz]"
mask_freq = select_mask(sky_freq, crtf_freq)

freq_selected = mask_freq.any(dim=["time", "polarization", "l", "m"])
print("Frequencies selected (GHz):", list(sky_freq.coords["frequency"].values[freq_selected.values] / 1e9))

plot_sky_with_mask(sky_freq, mask_freq.isel(frequency=2), title="frequency range 1.2–1.5 GHz, 1.2 GHz slice")

### Velocity range

`range=[v1, v2]` with units `km/s` or `m/s` selects channels by their velocity coordinate.
The DataArray must carry a `velocity` coordinate on the frequency axis.

In [ ]:
crtf_vel = "#CRTF\ncircle[[0arcsec, 0arcsec], 20arcsec], range=[20km/s, 50km/s]"
mask_vel = select_mask(sky_freq, crtf_vel)

freq_selected = mask_vel.any(dim=["time", "polarization", "l", "m"])
print("Velocities selected (km/s):", list(sky_freq.coords["velocity"].values[freq_selected.values] / 1e3))

plot_sky_with_mask(sky_freq, mask_vel.isel(frequency=2), title="velocity range 20–50 km/s, v=20km/s slice")

### Out-of-range spec → all-False mask

A `range=` that does not overlap the available channels returns an all-False mask.

In [ ]:
crtf_oor = "#CRTF\ncircle[[0arcsec, 0arcsec], 20arcsec], range=[5.0GHz, 6.0GHz]"
mask_oor = select_mask(sky_freq, crtf_oor)
print("Any pixel selected?", bool(mask_oor.values.any()))  # should be False

<a id="corr-selection"></a>
## `corr=` — polarization selection

The `corr=` keyword restricts the mask to one or more named Stokes / correlation products.
The values must match the `polarization` coordinate on the DataArray (case-insensitive).

`corr=` combines with a shape on the same line — the effective mask is the AND of the spatial
shape and the polarization selection.

In [ ]:
# 4-polarization sky (I, Q, U, V)
sky_pol = make_sky(pols=["I", "Q", "U", "V"])
print("Polarizations:", list(sky_pol.coords["polarization"].values))

### Select a single polarization

`corr=[I]` keeps only Stokes I.  The spatial mask (circle here) applies to that polarization.

In [ ]:
crtf_corr_i = "#CRTF\ncircle[[0arcsec, 0arcsec], 15arcsec], corr=[I]"
mask_corr_i = select_mask(sky_pol, crtf_corr_i)

pol_selected = mask_corr_i.any(dim=["time", "frequency", "l", "m"])
print("Polarizations selected:", list(sky_pol.coords["polarization"].values[pol_selected.values]))

plot_sky_with_mask(sky_pol, mask_corr_i.isel(polarization=0), title="corr=[I] — Stokes I only")

### Select multiple polarizations

`corr=[I, Q]` selects both Stokes I and Q.  U and V are excluded.

In [ ]:
crtf_corr_iq = "#CRTF\ncircle[[0arcsec, 0arcsec], 15arcsec], corr=[I, Q]"
mask_corr_iq = select_mask(sky_pol, crtf_corr_iq)

pol_selected = mask_corr_iq.any(dim=["time", "frequency", "l", "m"])
print("Polarizations selected:", list(sky_pol.coords["polarization"].values[pol_selected.values]))

# Verify U and V are excluded
np.testing.assert_array_equal(pol_selected.values, [True, True, False, False])
print("U and V correctly excluded ✓")

plot_sky_with_mask(sky_pol, mask_corr_iq.isel(polarization=0), title="corr=[I,Q] — spatial mask at I")

<a id="time-selection"></a>
## `time=` — time selection

The `time=` keyword restricts the mask to time steps whose MJD falls within the specified range.
Three token families are supported:

| Family | Example | Notes |
|---|---|---|
| MJD bare | `60001.0` | Modified Julian Day (days) |
| MJD with unit | `60001.0d` | Explicit `d` suffix |
| JD | `2460001.5jd` | Julian Day (must end in `jd`) |
| ISO | `'2023-03-01T00:00:00'` | ISO 8601, must be quoted |

Both endpoints must belong to the **same family** (mixing raises `ValueError`).
A non-overlapping range returns an **all-False** mask.

`time=` combines with a shape on the same line.

In [ ]:
# 5-step sky: times MJD 60000, 60001, ..., 60004
sky_time = make_sky(n_time=5, time_start_mjd=60000.0, time_step_mjd=1.0)
print("Time steps (MJD):", list(sky_time.coords["time"].values))

### MJD range

`time=[t1, t2]` with bare numbers selects time steps whose MJD falls within [t1, t2] (inclusive).

In [ ]:
crtf_mjd = "#CRTF\ncircle[[0arcsec, 0arcsec], 15arcsec], time=[60001.0, 60003.0]"
mask_mjd = select_mask(sky_time, crtf_mjd)

time_selected = mask_mjd.any(dim=["frequency", "polarization", "l", "m"])
print("Time steps selected (MJD):", list(sky_time.coords["time"].values[time_selected.values]))

plot_sky_with_mask(sky_time, mask_mjd.isel(time=1), title="MJD range 60001–60003, t=60001 slice")

### ISO format

ISO 8601 timestamps (quoted) are also accepted.  `astropy.time.Time` converts them to MJD
before comparing against the `time` coordinate.

In [ ]:
from astropy.time import Time

# Convert our MJD endpoints to ISO strings
lo_iso = Time(60001.0, format="mjd", scale="utc").isot
hi_iso = Time(60003.0, format="mjd", scale="utc").isot
print(f"ISO range: '{lo_iso}' to '{hi_iso}'")

crtf_iso = f"#CRTF\ncircle[[0arcsec, 0arcsec], 15arcsec], time=['{lo_iso}', '{hi_iso}']"
mask_iso = select_mask(sky_time, crtf_iso)

time_selected = mask_iso.any(dim=["frequency", "polarization", "l", "m"])
print("Time steps selected (MJD):", list(sky_time.coords["time"].values[time_selected.values]))

# MJD and ISO selections must agree
np.testing.assert_array_equal(mask_mjd.values, mask_iso.values)
print("MJD and ISO masks match ✓")

### Non-overlapping range → all-False mask

A `time=` range that does not cover any time step returns an all-False mask, rather than raising
an error.  This is the intended behaviour for pipeline use-cases where the selection spec is
fixed but the data window may not overlap.

In [ ]:
crtf_oot = "#CRTF\ncircle[[0arcsec, 0arcsec], 15arcsec], time=[59990.0, 59999.0]"
mask_oot = select_mask(sky_time, crtf_oot)
print("Any pixel selected?", bool(mask_oot.values.any()))   # should be False